# CBCTer GPU model runner (experimental)
Open in Google Colab, then **Runtime → Change runtime type → GPU**. Free GPUs are not guaranteed. This notebook has not been executed on a GPU by the CPU-only benchmark environment.

Choose **OralSeg** or **TIPs**. Upload a **Scan only**, native version-2 `.cbct.zip` exported by CBCTer. Colab processes the scan in your Google runtime; only upload data you are authorized to process there. Do not commit scan files, model weights or results to GitHub.

The downloads return two files: `.cbcter.zip` for **Review teeth → Import model result** on the matching full-grid scan, and `.cbct.zip` for direct streamed viewing on mobile or desktop. All proposals remain unreviewed. Model architecture/weights/dependency failures stop the run; a notebook is not evidence that a model passed scan tests.

In [ ]:
from pathlib import Path
import subprocess, sys
MODEL = "OralSeg" # or "TIPs"
assert MODEL in ["OralSeg", "TIPs"]
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)
subprocess.run(["uv", "venv", "--python", "3.10", "/content/cbct-gpu-env"], check=True)
PYTHON = "/content/cbct-gpu-env/bin/python"
def pip(*args): subprocess.run(["uv", "pip", "install", "--python", PYTHON, *args], check=True)
pip("torch==2.2.0", "--index-url", "https://download.pytorch.org/whl/cu121")
pip("numpy==1.26.4", "monai==1.4.0" if MODEL == "OralSeg" else "monai==1.3.0", "nibabel", "SimpleITK", "zstandard", "jcs==0.2.1", "huggingface-hub", "einops", "ninja", "packaging", "wheel", "setuptools", "gdown")
pip("causal-conv1d==1.4.0", "mamba-ssm==2.2.0", "--no-build-isolation")
subprocess.run([PYTHON, "-c", "import torch, mamba_ssm; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())"], check=True)


If the CUDA/Mamba installation fails, retain the error log. Colab CUDA images change; this pinned Python/Torch environment may need a matching Mamba wheel or CUDA toolkit. Do not substitute a CPU Mamba stub: it would change the model.

In [ ]:
import urllib.request
REPO = Path("/content/CBCTer")
if not REPO.exists(): subprocess.run(["git", "clone", "https://github.com/ahzs645/CBCTer", str(REPO)], check=True)
WORK = Path("/content/cbct-run"); WORK.mkdir(exist_ok=True)
from google.colab import files
uploaded = files.upload()
scan_names = [name for name in uploaded if name.endswith(".cbct.zip")]
assert len(scan_names) == 1, "Upload exactly one Scan only .cbct.zip"
SCAN = WORK / "source.cbct.zip"
SCAN.write_bytes(uploaded[scan_names[0]])
del uploaded
subprocess.run([PYTHON, str(REPO / "scripts/cbct_gpu_bridge.py"), "prepare", "--scan", str(SCAN), "--output", str(WORK)], check=True)


In [ ]:
if MODEL == "OralSeg":
    source = Path("/content/OralSeg")
    if not source.exists(): subprocess.run(["git", "clone", "https://github.com/OttoYouZhou/oralseg", str(source)], check=True)
    subprocess.run(["git", "-C", str(source), "checkout", "dd0eb05a97ed88e48da6f824fea2989bd4041c1c"], check=True)
    checkpoint = WORK / "model_workstation39.pt"
    urllib.request.urlretrieve("https://huggingface.co/aiadir/OralSeg/resolve/f5c60bf1aa05dfc791fdb07439f9141d7af10265/model_workstation39.pt", checkpoint)
    import hashlib
    with checkpoint.open("rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == "1b4b5806a0d2019d1b58b274cbb64c9b9f7f6510ddeff0413974acca58863ed2", "OralSeg checkpoint fingerprint changed"
    prediction = WORK / "oralseg-native.nii.gz"
    subprocess.run([PYTHON, str(REPO / "scripts/run_oralseg_gpu.py"), "--source", str(source), "--checkpoint", str(checkpoint), "--image", str(WORK / "scan.nii.gz"), "--output", str(prediction)], check=True)
else:
    source = Path("/content/TIPs")
    if not source.exists(): subprocess.run(["git", "clone", "https://github.com/TaoZhong11/TIPs", str(source)], check=True)
    subprocess.run(["git", "-C", str(source), "checkout", "e563c7f3477d47cba9486f5b3a47fad100c48b68"], check=True)
    pip("acvl-utils==0.2", "dynamic-network-architectures==0.2", "-e", str(source))
    weights_zip = WORK / "tips-models.zip"
    subprocess.run(["/content/cbct-gpu-env/bin/gdown", "1UuFgZ-kwRryPC-vK7w64xX0VO4iOAeGt", "-O", str(weights_zip)], check=True)
    import zipfile
    with zipfile.ZipFile(weights_zip) as archive:
        for item in archive.infolist():
            target = (source / item.filename).resolve()
            assert target.is_relative_to(source.resolve()), "Unsafe weights archive path"
        archive.extractall(source)
    assert (source / "nnResults").is_dir(), "Place the archive's nnResults folder at /content/TIPs/nnResults"
    import shutil
    inputs = WORK / "tips-input"; inputs.mkdir(exist_ok=True)
    shutil.copyfile(WORK / "scan.nii.gz", inputs / "scan.nii.gz")
    subprocess.run([PYTHON, str(source / "TIPs.py"), str(inputs)], cwd=source, check=True)
    prediction = Path(str(inputs) + "_resample_teeth_instance") / "scan.nii.gz"
    assert prediction.exists(), "TIPs did not produce tooth instances; inspect every inference stage log."


TIPs uses three models (binary teeth, tooth instances, pulp). Its upstream script does not propagate every subprocess failure. Check its stage logs and output volumes before using a result. This notebook packages the tooth-instance output; pulp files remain in `tips-input_resample_pulps_instance` for separate review. OralSeg weights are published under CC-BY-NC-4.0 (noncommercial). TIPs weight licensing needs verification before commercial deployment.

In [ ]:
import hashlib, json
if MODEL == "OralSeg":
    provenance = json.loads(Path(str(prediction) + ".json").read_text())
    weights_hash = provenance["weightsSha256"]
else:
    hashes = []
    for checkpoint in sorted((source / "nnResults").rglob("*.pth")):
        with checkpoint.open("rb") as f: hashes.append([str(checkpoint.relative_to(source)), hashlib.file_digest(f, "sha256").hexdigest()])
    assert hashes, "No TIPs checkpoints were found"
    weights_hash = hashlib.sha256(json.dumps(hashes, separators=(",", ":")).encode()).hexdigest()
    (WORK / "tips-weights-provenance.json").write_text(json.dumps(hashes, indent=2))
if MODEL == "TIPs":
    provenance = {"weightsSha256": weights_hash, "sourceCommit": "e563c7f3477d47cba9486f5b3a47fad100c48b68", "checkpointHashes": hashes, "pipeline": "upstream TIPs.py, datasets 803/812/810", "clinicalAccuracy": None}
provenance_path = WORK / "model-run-provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2))
result = WORK / (MODEL.lower() + "-result")
subprocess.run([PYTHON, str(REPO / "scripts/cbct_gpu_bridge.py"), "package", "--scan", str(SCAN), "--prediction", str(prediction), "--output", str(result), "--model", MODEL, "--weights-hash", weights_hash, "--provenance", str(provenance_path), "--label-scheme", "oralseg" if MODEL == "OralSeg" else "fdi"], check=True)
files.download(str(result) + ".cbcter.zip")
files.download(str(result) + ".cbct.zip")


In CBCTer: open the complete `.cbct.zip` to inspect streamed overlays. To combine with an existing case, load the matching native full-grid scan, open **Review teeth**, and import `.cbcter.zip`. A different/cropped scan is rejected. Large cases can exceed mobile editing memory; use streamed review or a desktop. Select a tooth, inspect all three planes, correct numbering/outlines, and explicitly accept only reviewed proposals. **Scan + analysis** export saves original predictions, corrections and provenance together.